# LLM が生成した文章を見分ける（パープレキシティ、Binoculars、分類器）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_llm_analysis/ai_text_detection.ipynb)

NLP クックブック 23-10 ／ LLM の評価と分析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

人が書いた文章と LLM が生成した文章を見分けたい。言語モデルの尤度にもとづく方法と、学習した分類器とを比べ、どんな条件で見分けられなくなるかも確かめる。

In [1]:
!pip install -q datasets sentence-transformers

In [2]:
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
L = 150   # 人の文章も生成文も、整形してから先頭 L 文字にそろえる (長さの違いで見分けてしまわないように)

def clean(text):
    """括弧書き (読みがな・外国語名など)、Markdown の記号、改行と空白を除く"""
    text = re.sub(r"（[^（）]*）|\([^()]*\)", "", text)
    text = re.sub(r"^\s*(#+|[-*・]|\d+\.)\s*", "", text, flags=re.MULTILINE)
    return re.sub(r"\*\*|\s+", "", text)

# 人の文章: 日本語版 Wikipedia の記事の冒頭 (CC BY-SA 3.0 / GFDL、データセットカードによる)。ダンプの先頭から順に 200 記事を使う
wiki = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)
human = []
for ex in wiki:
    if any(w in ex["title"] for w in ["一覧", "曖昧さ回避"]) or re.search(r"\d", ex["title"]):
        continue
    lead = clean(ex["text"].split("\n\n")[0])      # 最初の段落
    if len(lead) >= L:
        human.append({"title": ex["title"], "text": lead[:L]})
    if len(human) == 200:
        break
print(len(human), "件"); print(human[0])

200 件
{'title': '日本語', 'text': '日本語は、日本国内や、かつての日本領だった国、そして国外移民や移住者を含む日本人同士の間で使用されている言語。日本は法令によって公用語を規定していないが、法令その他の公用文は全て日本語で記述され、各種法令において日本語を用いることが規定され、学校教育においては「国語」の教科として学習を行うなど、事実'}


In [3]:
# AI の文章: 同じ見出し語について、この章の小型 LLM に百科事典風の説明を書かせる (Colab の T4 で 2〜3 分)
gen_name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # SB Intuitions、MIT ライセンス

def load_sarashina_tokenizer(name, **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(name, **kwargs)
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    return tokenizer

gen_tok = load_sarashina_tokenizer(gen_name, padding_side="left")
gen_model = AutoModelForCausalLM.from_pretrained(gen_name, dtype=DTYPE, device_map="cuda").eval()
PROMPT = "「{}」について、百科事典の記事の冒頭部分を書いてください。見出しや箇条書きは使わず、である調で、300 字程度の文章にしてください。"

@torch.no_grad()
def generate(model, tok, prompts, batch_size=32, max_new_tokens=220, **kwargs):
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    outs = []
    for i in range(0, len(prompts), batch_size):
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False)
                 for p in prompts[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, pad_token_id=tok.pad_token_id, **kwargs)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [clean(o) for o in outs]

titles = [h["title"] for h in human]
ai = generate(gen_model, gen_tok, [PROMPT.format(t) for t in titles], temperature=0.7, top_p=0.95)
keep = [i for i, a in enumerate(ai) if len(a) >= L]      # L 文字に満たない生成文は、その話題ごと除く
print(f"{len(keep)} / {len(ai)} 件を使う")
df = pd.DataFrame({"title": [titles[i] for i in keep], "human": [human[i]["text"] for i in keep], "ai": [ai[i][:L] for i in keep],
                   "ai_full": [ai[i] for i in keep]})   # 切り詰める前の生成文 (言い換えの入力に使う)
for r in df.head(2).itertuples():
    print(f"[{r.title}]\n 人: {r.human}\n AI: {r.ai}")

200 / 200 件を使う
[日本語]
 人: 日本語は、日本国内や、かつての日本領だった国、そして国外移民や移住者を含む日本人同士の間で使用されている言語。日本は法令によって公用語を規定していないが、法令その他の公用文は全て日本語で記述され、各種法令において日本語を用いることが規定され、学校教育においては「国語」の教科として学習を行うなど、事実
 AI: 日本語は日本を代表する言語であり、その歴史は古く、縄文時代から現代まで様々な変遷を遂げてきました。漢字とひらがな、カタカナが融合した独特の表記法を持ち、その豊かな表現力で多くの文化圏に影響を与えています。日本語の文法は時制や活用が豊富であり、特に動詞の活用形は数えきれないほど存在します。さらに、語彙
[コケ植物]
 人: コケ植物とは、維管束を持たず、胞子散布を行う、単相で有性の配偶体世代が優先する陸上植物の一群である。コケ類や蘚苔類、蘚苔植物などともいう。日本では1665種程度、世界中でおよそ2万種ほどが記録されている。植物体は、その形態により、葉と茎の区別がはっきりとした茎葉体および、区別が曖昧な葉状体に分けられ
 AI: 「コケ植物」は、陸上に現存する植物の中でも特にユニークなグループの一つです。その名称が示すように、主にmossと呼ばれる仲間で構成されています。コケ植物の主要な特徴は、多細胞ではなく単細胞であることです。これは、光合成を行うための葉緑素を持たず、代わりに菌類と共生関係にあることに由来します。コケ植物


In [4]:
# 尤度にもとづく 3 つのスコア。どれも「AI らしいほど大きい」向きにそろえる
obs_name = "sbintuitions/sarashina2.2-0.5b"     # 指示チューニング前のベースモデル (SB Intuitions、MIT ライセンス)
obs_model = AutoModelForCausalLM.from_pretrained(obs_name, dtype=DTYPE, device_map="cuda").eval()
perf_model = gen_model                           # Binoculars のもう一方のモデル (ここでは生成に使ったモデルと同じ)
tok = load_sarashina_tokenizer(obs_name)         # 2 つのモデルはトークナイザーが同じ

@torch.no_grad()
def scores(text):
    ids = tok(text, return_tensors="pt")["input_ids"].to("cuda")
    lp_obs = obs_model(ids).logits[0, :-1].float().log_softmax(-1)      # 位置 t の次のトークンの分布
    lp_perf = perf_model(ids).logits[0, :-1].float().log_softmax(-1)
    y = ids[0, 1:]
    ll = lp_obs.gather(1, y[:, None])[:, 0]                              # 実際のトークンの対数確率
    # (1) パープレキシティ: 平均対数尤度が高い (= 予測しやすい) ほど AI らしい
    ppl = ll.mean()
    # (2) Fast-DetectGPT (解析解の版): 実際の文の対数尤度が、モデル自身がサンプリングした場合の期待値より何標準偏差高いか
    #     (元の論文はサンプリング用と採点用に別のモデルも使えるが、ここでは同じベースモデルを使う)
    p = lp_obs.exp()
    mu, var = (p * lp_obs).sum(-1), (p * lp_obs ** 2).sum(-1) - (p * lp_obs).sum(-1) ** 2
    fast = (ll.sum() - mu.sum()) / var.sum().sqrt()
    # (3) Binoculars: 一方のモデルのパープレキシティを、2 つのモデルの予測の食い違い (交差パープレキシティ) で割る
    #     分子は instruct モデル、分母はベースモデルの分布で instruct モデルの対数確率を重み付けする (公式実装に合わせた。
    #     論文の式 (4) は分子を M1 と書いているが、本文と実装では分子に instruct モデルを使う)
    log_ppl = -lp_perf.gather(1, y[:, None]).mean()
    x_ppl = -(p * lp_perf).sum(-1).mean()
    bino = -(log_ppl / x_ppl)                                            # 元の論文では小さいほど AI。符号を反転する
    return {"perplexity": ppl.item(), "fast_detectgpt": fast.item(), "binoculars": bino.item()}

from sklearn.metrics import roc_auc_score

def score_df(texts):
    return pd.DataFrame([scores(t) for t in texts])

S_human, S_ai = score_df(df["human"]), score_df(df["ai"])
def auc(s_h, s_a):
    return {c: roc_auc_score([0] * len(s_h) + [1] * len(s_a), pd.concat([s_h[c], s_a[c]])) for c in s_h.columns}
print("ROC-AUC (全件):", {k: round(float(v), 3) for k, v in auc(S_human, S_ai).items()})
print(pd.concat({"人": S_human.mean(), "AI": S_ai.mean()}, axis=1).round(3).to_string())

ROC-AUC (全件): {'perplexity': 0.862, 'fast_detectgpt': 0.914, 'binoculars': 0.999}
                    人     AI
perplexity     -3.087 -2.309
fast_detectgpt -0.172  1.717
binoculars     -0.985 -0.569


In [5]:
# 学習する分類器: 話題で学習用と評価用に分け (同じ話題の人/AI の文は同じ側に入る)、文字 n-gram と文埋め込みで判定する
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
idx = rng.permutation(len(df)); tr, te = idx[: len(df) // 2], idx[len(df) // 2:]
X_tr = list(df["human"].iloc[tr]) + list(df["ai"].iloc[tr]); y_tr = [0] * len(tr) + [1] * len(tr)

ngram = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True), LogisticRegression(C=10, max_iter=1000)).fit(X_tr, y_tr)
e5 = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")   # Microsoft (intfloat)、MIT ライセンス
embed = lambda texts: e5.encode(["query: " + t for t in texts], normalize_embeddings=True, batch_size=64)
emb_clf = LogisticRegression(C=10, max_iter=1000).fit(embed(X_tr), y_tr)

# 文字 n-gram の分類器は何を手がかりにしているか
vocab, coef = ngram[0].get_feature_names_out(), ngram[-1].coef_[0]
print("AI 側の手がかり:", list(vocab[coef.argsort()[::-1][:12]]))
print("人側の手がかり:", list(vocab[coef.argsort()[:12]]))
print("「です」「ます」を含む割合: 人", df["human"].str.contains("です|ます").mean(), "/ AI", round(df["ai"].str.contains("です|ます").mean(), 2))

def all_scores(texts, rows):
    """評価用の行 rows について、5 つの方法のスコアを返す"""
    texts = list(pd.Series(texts).iloc[rows])
    s = score_df(texts)
    s["char_ngram"] = ngram.predict_proba(texts)[:, 1]
    s["embedding"] = emb_clf.predict_proba(embed(texts))[:, 1]
    return s

H, A = all_scores(df["human"], te), all_scores(df["ai"], te)
results = {"基本 (sarashina 0.5B, 温度 0.7)": auc(H, A) | {"件数": len(te)}}
print(f"評価: 人 {len(te)} 件 / AI {len(te)} 件")
print(pd.DataFrame(results).round(3).to_string())

AI 側の手がかり: ['す。', 'ます', 'ます。', 'ま', 'いま', 'います', 'です', 'です。', 'ていま', 'す', 'しま', '特']
人側の手がかり: ['る。', 'ある。', 'ある', 'である', '・', 'いる', 'ている', 'れる。', 'る', 'また', '0', 'れる']
「です」「ます」を含む割合: 人 0.0 / AI 0.99
評価: 人 100 件 / AI 100 件
                基本 (sarashina 0.5B, 温度 0.7)
perplexity                            0.877
fast_detectgpt                        0.921
binoculars                            1.000
char_ngram                            1.000
embedding                             0.991
件数                                  100.000


In [6]:
# ブートストラップで ROC-AUC の 95% 信頼区間を求める (評価は 100 件程度なので幅がある)
def auc_ci(c, n=1000):
    vals = []
    for _ in range(n):
        i, j = rng.integers(0, len(H), len(H)), rng.integers(0, len(A), len(A))
        vals.append(roc_auc_score([0] * len(i) + [1] * len(j), np.r_[H[c].values[i], A[c].values[j]]))
    return np.percentile(vals, [2.5, 97.5]).round(3).tolist()
print({c: auc_ci(c) for c in H.columns})

{'perplexity': [0.827, 0.926], 'fast_detectgpt': [0.883, 0.959], 'binoculars': [0.999, 1.0], 'char_ngram': [0.998, 1.0], 'embedding': [0.982, 0.997]}


In [7]:
# 限界: 生成の条件を変えると見分けにくくなる (分類器は「基本」の条件で学習したまま。Colab の T4 で 3〜4 分)
titles_te = [df["title"].iloc[i] for i in te]
hot = generate(gen_model, gen_tok, [PROMPT.format(t) for t in titles_te], temperature=1.0, top_p=1.0)

other_name = "Qwen/Qwen2.5-1.5B-Instruct"       # 別のモデル (Alibaba の Qwen チーム、Apache 2.0)
other_tok = AutoTokenizer.from_pretrained(other_name, padding_side="left")
other_model = AutoModelForCausalLM.from_pretrained(other_name, dtype=DTYPE, device_map="cuda").eval()
other = generate(other_model, other_tok, [PROMPT.format(t) for t in titles_te], temperature=0.7, top_p=0.95)

# 言い換え: 基本の生成文 (切り詰める前) を、別のモデルに書き直させる
para = generate(other_model, other_tok, ["次の文章を、意味を変えずに別の言い回しで書き直してください。である調で、書き直した文章だけを出力してください。\n\n" + df["ai_full"].iloc[i] for i in te],
                temperature=0.7, top_p=0.95)

# 文体 (です・ます) と、英字の混入 (3 文字以上の連続) の割合も見る。尤度の方法も分類器も、これらの影響を受けうる
style = lambda texts: {"です・ます": np.mean([bool(re.search("です|ます", t)) for t in texts]), "英字の混入": np.mean([bool(re.search("[A-Za-z]{3,}", t)) for t in texts])}
styles = {"人": style(df["human"].iloc[te]), "基本 (sarashina 0.5B, 温度 0.7)": style(df["ai"].iloc[te])}
for label, texts in [("温度 1.0", hot), ("別のモデル (Qwen2.5 1.5B)", other), ("言い換え (Qwen2.5 1.5B)", para)]:
    texts = [t[:L] for t in texts if len(t) >= L]
    results[label] = auc(H, all_scores(texts, list(range(len(texts))))) | {"件数": len(texts)}
    styles[label] = style(texts)
    print(f"[{label}] {texts[0]}")
print(pd.DataFrame(results).round(3).to_string())
print(pd.DataFrame(styles).round(2).to_string())

[温度 1.0] 「リーグ戦」とは、複数のチームが定期的に対戦し競争を競う組織化されたmatchesのことです。このリーグ制は一般的なスポーツだけでなく、テニスやラグビー、バスケットボールなど様々な分野で見られます。各チームは特定の期間中に一定のラウンドを戦い抜き、最終的な順位を決定します。このプロセスにより、各チー
[別のモデル (Qwen2.5 1.5B)] 日本語版Wikipediaでは、「リーグ戦」については以下のようになっています：リーグ戦シーズン中に各チームが行う正式な試合現在、代表選手権大会以外の主なスポーツにおいて行われる。主にホームタウンでの開催が多く、地元ファンへの応援が強くなる。毎年同じメンバーによる固定的な試合形式。各チームの成績、得
[言い換え (Qwen2.5 1.5B)] リーグ戦は、定期的に各チーム間で行われるスポーツの試合をまとめたものです。これは選手たちの成長と戦略練習の場でもあり、ファンには新たな喜びと満足感を与えます。また、各チームが一定回数の試合を行いながら順位や勝敗を競うことで、チーム間での競争と連携が進みます。さらに、上位チームと下位チームの交流を通じ
                基本 (sarashina 0.5B, 温度 0.7)   温度 1.0  別のモデル (Qwen2.5 1.5B)  言い換え (Qwen2.5 1.5B)
perplexity                            0.877    0.566                 0.455                0.546
fast_detectgpt                        0.921    0.580                 0.440                0.540
binoculars                            1.000    0.988                 0.715                0.892
char_ngram                            1.000    0.999                 0.988                0.999
embedding    

In [8]:
# 人の文章を AI と誤判定する危険: 人の文章の 5% を誤判定するしきい値で、定型的な文を判定してみる
templates = [   # どれも筆者が書いた人の文章 (150 字程度にそろえてある)
    "本日はお忙しいところ、ご来場いただき誠にありがとうございます。開会にあたり、主催者を代表して一言ご挨拶申し上げます。本大会は今年で十回目を迎えることとなりました。これもひとえに、関係者の皆様の温かいご支援とご協力のおかげであり、心より御礼申し上げます。本日一日が、皆様にとって実りあるものとなりますよう願っております。",
    "平素より格別のお引き立てを賜り、誠に厚く御礼申し上げます。さて、このたび弊社は業務拡大に伴い、下記の住所へ事務所を移転いたしましたので、お知らせいたします。これを機に、社員一同さらなるサービスの向上に努めてまいる所存でございます。今後とも変わらぬご愛顧を賜りますよう、何卒よろしくお願い申し上げます。",
    "本規約は、当社が提供するサービスの利用条件を定めるものです。利用者は、本規約に同意したうえで本サービスを利用するものとします。当社は、利用者に事前に通知することなく、本規約を変更できるものとします。変更後の規約は、当社のウェブサイトに掲載した時点から効力を生じるものとします。利用者は、定期的に本規約を確認するものとします。",
    "私は毎日学校に行きます。学校で友達と話します。昼ごはんを食べます。午後は勉強します。夜は家で宿題をします。週末は家族と買い物に行きます。私は日本の料理が好きです。特にラーメンが好きです。日本語の勉強は難しいですが、楽しいです。毎日少しずつ漢字を覚えています。来年は日本の大学に入りたいです。がんばります。",
    "きのう、駅前の古い喫茶店で、十年ぶりに高校の同級生と会った。互いの白髪を笑いあううちに、閉店の時間になっていた。店を出ると、雨はもうやんでいて、濡れた石畳に街灯の光が細く伸びていた。またすぐ会おうと言って別れたが、次に会うのはきっとまた何年も先なのだろう。それでもいいと思えるのが、少しだけうれしかった。",
]
templates = [t[:L] for t in templates]
T = all_scores(templates, list(range(len(templates))))
print("文字数:", [len(t) for t in templates])
th = H.quantile(0.95)
print("AI と判定されたか (しきい値は人の Wikipedia の文の 95 パーセンタイル):")
print(pd.DataFrame(T > th).replace({True: "AI", False: "人"}).set_index(pd.Index([t[:15] for t in templates])).to_string())

文字数: [150, 150, 150, 150, 150]
AI と判定されたか (しきい値は人の Wikipedia の文の 95 パーセンタイル):
                perplexity fast_detectgpt binoculars char_ngram embedding
本日はお忙しいところ、ご来場い          人              人          人         AI        AI
平素より格別のお引き立てを賜り         AI             AI         AI         AI        AI
本規約は、当社が提供するサービ         AI             AI         AI         AI        AI
私は毎日学校に行きます。学校で          人              人          人         AI        AI
きのう、駅前の古い喫茶店で、十          人              人          人          人         人


- 尤度にもとづく方法は学習が要らない（ゼロショット）。パープレキシティ（ここでは平均対数尤度）は「LLM の文は言語モデルにとって予測しやすい」という手がかりを使う。DetectGPT (Mitchell et al., 2023) は「文を少し書き換えると尤度が下がるなら AI」と判定するが、書き換えに別のモデルを何十回も使うので重い。Fast-DetectGPT (Bao et al., 2024) はこれを 1 回の順伝播で計算できる形にしたもので、ここでは解析解の版を実装した。Binoculars (Hans et al., 2024) は、2 つのモデルの予測の食い違いで割ることで、話題による尤度の違いを打ち消す。
- この実行の ROC-AUC（評価は人 100 件・AI 100 件）は、パープレキシティ 0.877（95% 信頼区間 0.83〜0.93）、Fast-DetectGPT 0.921 (0.88〜0.96)、Binoculars 1.000 だった。ただし Binoculars の一方のモデルは生成に使ったモデルそのものなので、有利な設定である。別のモデル (Qwen2.5 1.5B) の文では Binoculars も 0.715 に下がった。検出に使うモデルと生成したモデルが違うと弱くなる。信頼区間はブートストラップで求めた（評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照）。
- 分類器の ROC-AUC もほぼ 1 だが、文字 n-gram の手がかりの上位は「す。」「ます」「です」だった。「である調で」と指示しても小さなモデルはほとんど従わず、「です」「ます」を含む割合は、Wikipedia の文が 0、AI の文はどの条件でも 0.94〜1.00 だった。分類器は「AI らしさ」ではなく「Wikipedia らしくなさ」を学んでいて、条件を変えても AUC が高いままなのはそのためである。尤度にもとづく方法も、文体の違いの影響を受けうる。実際に使う場面と同じ種類の人の文章を学習・評価に入れる。
- 生成の条件を変えると、尤度にもとづく方法は大きく崩れた。温度 1.0 では、パープレキシティが 0.566、Fast-DetectGPT が 0.580 になった（温度 1.0 の文は英字の混じる割合が 0.66 と高く、崩れた文も多い）。別のモデルの文では、それぞれ 0.455 と 0.440 で、0.5 を下回った（人の文のほうが AI らしく見える）。言い換え（Qwen2.5 1.5B に書き直させたもの）では、それぞれ 0.546 と 0.540、Binoculars も 0.892 に下がった。各条件 100 件なので AUC には ±0.05 程度の幅がある。Krishna et al. (2023) は、言い換え用のモデルで書き直すだけで、DetectGPT を含む複数の検出器をすり抜けられることを示している。
- 人の文章を AI と誤判定する危険がある。Wikipedia の人の文の 5% を誤判定するしきい値で、筆者が書いた 150 字の文 5 つを判定すると、事務所移転の通知と利用規約のような定型文は、5 つの方法すべてで AI と判定された。やさしい語彙の短い文の連続（日本語学習者の作文を模したもの）と挨拶文は、2 つの分類器だけが AI と判定した。これは「です・ます」や敬語で書かれているためである。英語では、非母語話者の作文が AI と誤判定されやすいことが報告されている (Liang et al., 2023)。数例の結果だが、検出器の出力だけを根拠に、人が不正をしたと判断してはいけない。
- 人の文章には Wikipedia のダンプの先頭 200 記事を使った。よく知られた項目が多く、Wikipedia は多くの言語モデルの学習データに含まれるので、モデルが記憶していて人の文なのに尤度が高く出ることがある。長さや括弧書きのような表面的な違いで見分けないよう、両方から括弧書きを除き、同じ文字数にそろえた。
- sarashina2.2 のトークナイザーは、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きのとおりに直して使う。生成する側が協力できるなら、この章の「LLM の出力に透かしを入れて検出する（transformers の WatermarkingConfig と SynthID）」のように、生成時に透かしを埋め込み、後から検出する方法もある。